


#1. Problem

We have

$$
u''(x) = e^{\sin x}, \quad u'(0) = 0, \quad u'(1) = \alpha.
$$

We first integrate once:

$$
u'(x) = \int e^{\sin t} \, dt + C_1.
$$

Let’s denote
$$
F(x) = \int_0^x e^{\sin t} \, dt.
$$

Then

$$
u'(x) = F(x) + C_1.
$$



#2. Apply first boundary condition

$$
u'(0) = F(0) + C_1 = 0 + C_1 = 0 \implies C_1 = 0.
$$

So

$$
u'(x) = F(x) = \int_0^x e^{\sin t} \, dt.
$$



#3. Apply second boundary condition $(u'(1) = \alpha)$

$$
u'(1) = \int_0^1 e^{\sin t} \, dt = \alpha.
$$

So **for a solution to exist**, we must have

$$
\alpha = \int_0^1 e^{\sin x} \, dx.
$$

Let’s compute that integral (numerically, by hand-approximation):


We can use Simpson’s rule with $(n=4)$ subintervals: $(h = 0.25).$

Values:

$$(x_0 = 0): (e^{\sin 0} = e^0 = 1)$$
$$(x_1 = 0.25): (\sin 0.25 \approx 0.2474), (e^{0.2474} \approx 1.280)$$
$$(x_2 = 0.5): (\sin 0.5 \approx 0.4794), (e^{0.4794} \approx 1.615)$$
$$(x_3 = 0.75): (\sin 0.75 \approx 0.6816), (e^{0.6816} \approx 1.977)$$
$$(x_4 = 1.0): (\sin 1 \approx 0.84147), (e^{0.84147} \approx 2.319)$$

Simpson’s rule:

$$
\int_0^1 f(x) dx \approx \frac{h}{3} \left[ f_0 + f_4 + 4(f_1 + f_3) + 2 f_2 \right]
$$
$$
= \frac{0.25}{3} \left[ 1 + 2.319 + 4(1.280 + 1.977) + 2(1.615) \right]
$$
$$
= \frac{0.25}{3} \left[ 3.319 + 4(3.257) + 3.230 \right]
$$
$$
= \frac{0.25}{3} \left[ 3.319 + 13.028 + 3.230 \right]
$$
$$
= \frac{0.25}{3} \times 19.577
$$
$$
\approx 0.25 \times 6.5257 \approx 1.6314.
$$

So $(\alpha \approx 1.6314)$.



Thus:

$$
\boxed{\alpha = \int_0^1 e^{\sin x} \, dx}
$$
Numerically $(\alpha \approx 1.6314)$.





In [4]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Define RHS
# ------------------------------------------------------------
def f(x):
    return np.exp(np.sin(x))

# Required alpha for solvability (self-coded trapz)
xx = np.linspace(0,1,20000)
alpha_exact = np.trapz(f(xx), xx)


# ------------------------------------------------------------
# 2nd order FD solver with u(0)=0 and Neumann at x=1
# ------------------------------------------------------------
def solve_bvp(N, alpha):

    h = 1/N
    x = np.linspace(0, 1, N+1)

    A = np.zeros((N+1, N+1))
    b = np.zeros(N+1)

    # Enforce u(0)=0
    A[0,0] = 1.0
    b[0] = 0.0

    # interior points
    for i in range(1, N):
        A[i, i-1] = 1.0
        A[i, i]   = -2.0
        A[i, i+1] = 1.0
        b[i] = h*h * f(x[i])

    # Neumann at x=1: (u_N - u_{N-1})/h = alpha
    A[N, N]   =  1.0/h
    A[N, N-1] = -1.0/h
    b[N] = alpha

    u = np.linalg.solve(A, b)
    return x, u


# ------------------------------------------------------------
# Convergence test
# ------------------------------------------------------------
Ns = [20, 40, 80, 160, 320]
errors = []

def residual(x, u):
    h = x[1] - x[0]
    N = len(x)-1
    R = np.zeros_like(u)

    # PDE residual at interior nodes
    for i in range(1, N):
        R[i] = (u[i-1] - 2*u[i] + u[i+1])/(h*h) - f(x[i])

    # boundary residual at x=1
    R[0] = u[0]                    # should be 0
    R[N] = (u[N] - u[N-1])/h - alpha_exact

    return np.max(np.abs(R))


print("alpha_exact =", alpha_exact)

for N in Ns:
    x, u = solve_bvp(N, alpha_exact)
    R = residual(x, u)
    errors.append(R)
    print(f"N={N:4d}   max residual = {R:.2e}")




alpha_exact = 1.6318696084708444
N=  20   max residual = 3.20e-14
N=  40   max residual = 1.65e-13
N=  80   max residual = 1.13e-12
N= 160   max residual = 4.27e-12
N= 320   max residual = 2.36e-11


/tmp/ipython-input-110090597.py:12: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  alpha_exact = np.trapz(f(xx), xx)
